In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
from collections import deque
from typing import Dict, Tuple, List
import matplotlib.pyplot as plt
from PIL import Image
import io
from typing import Dict, Tuple, List

In [3]:
class SwitchEnv:
    def __init__(self, size=5):
        self.size = size
        self.num_agents = 2
        # ゴール（荷物の目的地）: エージェント0は右端(4)、エージェント1は左端(0)
        self.pickup_pos = 2  # 通路の真ん中
        self.targets = {0: 4, 1: 0}
        self.reset()

    def reset(self):
        # エージェントの初期位置: 両端
        self.agent_positions = {0: 0, 1: 4}
        self.agent_holding = {0: False, 1: False}
        self.done_delivery = {0: False, 1: False}
        self.steps = 0
        return self._get_obs()

    def _get_obs(self):
        obs = {}
        for i in range(self.num_agents):
            other = 1 - i
            # (自分の位置, 自分が持っているか, 相方の位置, 相方が持っているか)
            obs[i] = (
                self.agent_positions[i],
                self.agent_holding[i],
                self.agent_positions[other],
                self.agent_holding[other]
            )
        return obs

    def step(self, actions: Dict[int, int]):
        self.steps += 1
        rewards = {i: -0.1 for i in range(self.num_agents)} # 時間経過ペナルティ

        # 移動処理 (0: Stay, 1: Left, 2: Right)
        next_pos = {}
        for i, action in actions.items():
            p = self.agent_positions[i]
            if action == 1: p = max(0, p - 1)
            elif action == 2: p = min(self.size - 1, p + 1)
            next_pos[i] = p

        # 衝突判定（同じマスに入ろうとしたら移動失敗）
        if next_pos[0] == next_pos[1]:
            rewards[0] -= 1.0
            rewards[1] -= 1.0
            # 位置は更新しない
        else:
            self.agent_positions = next_pos

        # ピックアップ & ドロップオフ判定
        for i in range(self.num_agents):
            # 1. 荷物を拾う (真ん中にいて、誰も持っていない場合)
            if self.agent_positions[i] == self.pickup_pos and not any(self.agent_holding.values()) and not any(self.done_delivery.values()):
                self.agent_holding[i] = True
                rewards[i] += 5.0

            # 2. 届ける (自分のターゲットに到着)
            if self.agent_holding[i] and self.agent_positions[i] == self.targets[i]:
                self.agent_holding[i] = False
                self.done_delivery[i] = True
                rewards[i] += 20.0

        done = {i: any(self.done_delivery.values()) or self.steps >= 50 for i in range(self.num_agents)}
        return self._get_obs(), rewards, done, {}

    def render_frame(self):
        fig, ax = plt.subplots(figsize=(8, 2))
        ax.set_xlim(-0.5, self.size - 0.5)
        ax.set_ylim(-0.5, 0.5)
        ax.set_xticks(range(self.size))
        ax.set_yticks([])
        ax.grid(True)

        # ピックアップ地点
        ax.add_patch(plt.Rectangle((self.pickup_pos-0.4, -0.4), 0.8, 0.8, color='gray', alpha=0.2))
        ax.text(self.pickup_pos, 0.45, "Pickup", ha='center')

        # エージェント描画
        colors = {0: 'red', 1: 'blue'}
        for i in range(self.num_agents):
            pos = self.agent_positions[i]
            color = colors[i]
            marker = 's' if self.agent_holding[i] else 'o'
            ax.plot(pos, 0, marker, markersize=20, color=color, label=f"Agent {i}")
            # ターゲット地点の印
            ax.plot(self.targets[i], 0, 'x', markersize=12, color=color)

        ax.set_title(f"Step: {self.steps}")

        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        plt.close(fig)
        buf.seek(0)
        return Image.open(buf)



def save_random_behavior_gif(env, filename="random_behavior.gif", max_steps=30):
    frames = []
    obs = env.reset()

    print(f"🎬 Generating random behavior GIF: {filename}...")

    for t in range(max_steps):
        # 1. 現在のフレームをキャプチャしてリストに追加
        frame = env.render_frame()
        frames.append(frame)

        # 2. 全エージェントに対して完全にランダムな行動を選択
        # 0: Stay, 1: Left, 2: Right
        actions = {i: random.randint(0, 2) for i in range(env.num_agents)}

        # 3. 環境を1ステップ進める
        next_obs, rewards, done, info = env.step(actions)

        # デバッグ用に報酬を表示（任意）
        # print(f"Step {t}: Actions {actions}, Rewards {rewards}")

        if any(done.values()):
            # 終了（ゴールまたはタイムアップ）した場合は最後のフレームを撮って終了
            frames.append(env.render_frame())
            print(f"🏁 Episode finished at step {t}")
            break

    # 4. PIL Imageの機能を使ってGIFとして保存
    if frames:
        frames[0].save(
            filename,
            save_all=True,
            append_images=frames[1:],
            duration=300,  # ランダムな動きが見やすいよう少しゆっくり（0.3秒間隔）
            loop=0
        )
        print(f"✅ GIF saved successfully: {filename}")
    else:
        print("❌ No frames were captured.")

In [4]:

# --- ハイパーパラメータ ---
GRID_SIZE = 5
OBS_SHAPE = 4    # (self_pos, self_hold, other_pos, other_hold)
STATE_SHAPE = 8  # 全エージェントの情報の統合
N_ACTIONS = 3    # 0: Stay, 1: Left, 2: Right
BATCH_SIZE = 32
GAMMA = 0.95
LR = 5e-4
MEMORY_CAPACITY = 10000
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 0.995
TARGET_UPDATE_INTERVAL = 10
NUM_EPISODES = 500

In [5]:

# --- ネットワーク定義 ---

class MLPAgent(nn.Module):
    def __init__(self, input_shape, hidden_dim, n_actions):
        super(MLPAgent, self).__init__()
        self.fc1 = nn.Linear(input_shape, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, n_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

class QMixer(nn.Module):
    def __init__(self, n_agents, state_shape, mixing_embed_dim, hypernet_embed_dim):
        super(QMixer, self).__init__()
        self.n_agents = n_agents
        self.state_shape = state_shape

        # ハイパーネットワーク: 状態からQMixerの重みを生成
        self.hyper_w1 = nn.Sequential(nn.Linear(state_shape, hypernet_embed_dim),
                                      nn.ReLU(),
                                      nn.Linear(hypernet_embed_dim, n_agents * mixing_embed_dim))
        self.hyper_w2 = nn.Sequential(nn.Linear(state_shape, hypernet_embed_dim),
                                      nn.ReLU(),
                                      nn.Linear(hypernet_embed_dim, mixing_embed_dim))

        self.hyper_b1 = nn.Linear(state_shape, mixing_embed_dim)
        self.hyper_b2 = nn.Sequential(nn.Linear(state_shape, mixing_embed_dim),
                                      nn.ReLU(),
                                      nn.Linear(mixing_embed_dim, 1))

    def forward(self, agent_qs, states):
        bs = agent_qs.size(0)
        states = states.view(-1, self.state_shape)
        agent_qs = agent_qs.view(-1, 1, self.n_agents)

        # 重みは常に正（単調性）を保証するためにabsを取る
        w1 = torch.abs(self.hyper_w1(states)).view(-1, self.n_agents, 32)
        b1 = self.hyper_b1(states).view(-1, 1, 32)
        hidden = F.elu(torch.matmul(agent_qs, w1) + b1)

        w2 = torch.abs(self.hyper_w2(states)).view(-1, 32, 1)
        b2 = self.hyper_b2(states).view(-1, 1, 1)
        q_tot = torch.matmul(hidden, w2) + b2
        return q_tot.view(bs, -1)

# --- エージェントクラス ---

class QMixTrainer:
    def __init__(self, env):
        self.env = env
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.agent_net = MLPAgent(OBS_SHAPE, 64, N_ACTIONS).to(self.device)
        self.mixer_net = QMixer(2, STATE_SHAPE, 32, 64).to(self.device)

        self.target_agent_net = MLPAgent(OBS_SHAPE, 64, N_ACTIONS).to(self.device)
        self.target_mixer_net = QMixer(2, STATE_SHAPE, 32, 64).to(self.device)
        self.target_agent_net.load_state_dict(self.agent_net.state_dict())
        self.target_mixer_net.load_state_dict(self.mixer_net.state_dict())

        self.optimizer = optim.Adam(list(self.agent_net.parameters()) + list(self.mixer_net.parameters()), lr=LR)
        self.memory = deque(maxlen=MEMORY_CAPACITY)

    def _get_tensors(self, obs, is_state=False):
        # 観測を正規化してテンソル化
        if is_state:
            state = []
            for i in range(2):
                state.extend([obs[i][0]/4.0, 1.0 if obs[i][1] else 0.0, obs[i][2]/4.0, 1.0 if obs[i][3] else 0.0])
            return torch.FloatTensor(state).to(self.device).unsqueeze(0)
        else:
            tensors = {}
            for i in range(2):
                o = obs[i]
                tensors[i] = torch.FloatTensor([o[0]/4.0, 1.0 if o[1] else 0.0, o[2]/4.0, 1.0 if o[3] else 0.0]).to(self.device).unsqueeze(0)
            return tensors

    def select_actions(self, obs, epsilon):
        if random.random() < epsilon:
            return {i: random.randint(0, N_ACTIONS-1) for i in range(2)}

        tensors = self._get_tensors(obs)
        actions = {}
        with torch.no_grad():
            for i in range(2):
                q_values = self.agent_net(tensors[i])
                actions[i] = q_values.argmax().item()
        return actions

    def train_step(self):
        if len(self.memory) < BATCH_SIZE: return 0

        batch = random.sample(self.memory, BATCH_SIZE)
        s, a, r, s_next, d = zip(*batch)

        states = torch.cat([self._get_tensors(obs, True) for obs in s])
        next_states = torch.cat([self._get_tensors(obs, True) for obs in s_next])
        rewards = torch.FloatTensor(r).to(self.device).unsqueeze(1)
        dones = torch.FloatTensor(d).to(self.device).unsqueeze(1)

        # 現在のQ値計算
        agent_qs = []
        for i in range(2):
            obs_i = torch.cat([self._get_tensors(obs)[i] for obs in s])
            q_vals = self.agent_net(obs_i)
            act_i = torch.LongTensor([action[i] for action in a]).to(self.device).unsqueeze(1)
            agent_qs.append(q_vals.gather(1, act_i))

        q_tot = self.mixer_net(torch.cat(agent_qs, dim=1), states)

        # ターゲットQ値計算
        with torch.no_grad():
            target_agent_qs = []
            for i in range(2):
                next_obs_i = torch.cat([self._get_tensors(obs)[i] for obs in s_next])
                target_q_vals = self.target_agent_net(next_obs_i)
                target_agent_qs.append(target_q_vals.max(1)[0].unsqueeze(1))

            target_q_tot = self.target_mixer_net(torch.cat(target_agent_qs, dim=1), next_states)
            y = rewards + GAMMA * target_q_tot * (1 - dones)

        loss = F.mse_loss(q_tot, y)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()


In [6]:

# --- 学習メインループ ---

env = SwitchEnv(size=5)
trainer = QMixTrainer(env)
epsilon = EPS_START

print("🚀 学習開始...")
for ep in range(NUM_EPISODES):
    obs = env.reset()
    total_reward = 0
    done = False

    while not done:
        actions = trainer.select_actions(obs, epsilon)
        next_obs, rewards, dones, _ = env.step(actions)

        # QMIXはチーム報酬の合計を学習対象にする
        team_reward = sum(rewards.values())
        trainer.memory.append((obs, actions, team_reward, next_obs, any(dones.values())))

        obs = next_obs
        total_reward += team_reward
        done = any(dones.values())
        trainer.train_step()

    epsilon = max(EPS_END, epsilon * EPS_DECAY)

    if ep % TARGET_UPDATE_INTERVAL == 0:
        trainer.target_agent_net.load_state_dict(trainer.agent_net.state_dict())
        trainer.target_mixer_net.load_state_dict(trainer.mixer_net.state_dict())

    if ep % 50 == 0:
        print(f"Episode {ep} | Reward: {total_reward:.2f} | Epsilon: {epsilon:.3f}")



🚀 学習開始...
Episode 0 | Reward: -17.00 | Epsilon: 0.995
Episode 50 | Reward: 23.40 | Epsilon: 0.774
Episode 100 | Reward: 14.00 | Epsilon: 0.603
Episode 150 | Reward: 20.20 | Epsilon: 0.469
Episode 200 | Reward: 24.20 | Epsilon: 0.365
Episode 250 | Reward: 23.80 | Epsilon: 0.284
Episode 300 | Reward: 24.00 | Epsilon: 0.221
Episode 350 | Reward: 24.00 | Epsilon: 0.172
Episode 400 | Reward: 24.20 | Epsilon: 0.134
Episode 450 | Reward: 24.20 | Epsilon: 0.104


In [11]:
def save_agent_behavior_gif(agent, env, filename="agent_behavior.gif", max_steps=100):
    frames = []
    obs = env.reset()

    print("🎬 Generating frames for GIF...")

    for t in range(max_steps):
        frame = env.render_frame()

        # --- ここを修正：もし numpy配列なら PIL画像に変換する ---
        if isinstance(frame, np.ndarray):
            # もし [0, 1] の範囲なら 255倍するなどの処理が必要な場合があります
            if frame.max() <= 1.0:
                frame = (frame * 255).astype(np.uint8)
            frame = Image.fromarray(frame)
        # --------------------------------------------------

        frames.append(frame)

        actions = agent.select_actions(obs, epsilon=0.0)
        next_obs, rewards, done, info = env.step(actions)
        obs = next_obs

        if all(done.values()):
            # 最後のフレーム処理
            last_frame = env.render_frame()
            if isinstance(last_frame, np.ndarray):
                last_frame = Image.fromarray((last_frame * 255).astype(np.uint8)) if last_frame.max() <= 1.0 else Image.fromarray(last_frame)
            frames.append(last_frame)
            print(f"✅ Goal reached in {t} steps!")
            break

    if frames:
        # frames[0] が確実に PIL Image になっているので save が使えます
        frames[0].save(
            filename,
            save_all=True,
            append_images=frames[1:],
            duration=200,
            loop=0
        )
        print(f"💾 GIF saved as {filename}")

In [12]:
save_agent_behavior_gif(trainer, env, "switch_task_success.gif")

🎬 Generating frames for GIF...
✅ Goal reached in 3 steps!
💾 GIF saved as switch_task_success.gif
